# Multi-Representation Search: Step-by-Step Build-Up

A document is rarely well-represented by a single embedding. A research paper has a title, an abstract, body chunks, and category tags, each carrying a different signal. Treat all four as one dense vector and the title gets averaged out; chunk-level grounding for downstream reasoning disappears.

This notebook builds a Qdrant retrieval pipeline that uses each representation deliberately. Over six steps you'll go from a naive dense-only baseline to a fully fused pipeline with four named-vector prefetches, Reciprocal Rank Fusion, document-level grouping, and optional formula-based score boosting. After each step you'll run the same query and see the top retrieved papers change.

The design rationale (why each component is there, when to use it, when not to) lives in the accompanying [tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/multi-representation-search/). This notebook focuses on running the code and watching the result list shift.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/qdrant/examples/blob/master/multi-representation-search/multi-representation-search.ipynb)


## Requirements

This notebook uses [Qdrant Cloud Inference](https://qdrant.tech/documentation/inference/#qdrant-cloud-inference) to generate embeddings server-side, so no client-side embedding library is required. The free tier covers this notebook's footprint. Core BM25 runs on any Qdrant instance, but dense Cloud Inference is Cloud-only. To self-host, generate dense vectors on the client with a library like [FastEmbed](https://qdrant.tech/documentation/fastembed/) and pass them as raw vectors instead of `models.Document`.


In [ ]:
!pip install qdrant-client datasets

## Dataset

20 000 ML/CS arXiv papers (2018 and later) from the [`gfissore/arxiv-abstracts-2021`](https://huggingface.co/datasets/gfissore/arxiv-abstracts-2021) dataset. Each paper has a `title`, `abstract`, and `categories` (which this dataset returns as space-joined strings, so we split them before filtering).


In [ ]:
from datasets import load_dataset

ML_CATEGORIES = {"cs.LG", "cs.CV", "cs.CL", "cs.AI", "stat.ML"}

# Non-streaming so HF caches the parquet locally; first run downloads ~2.5 GB, re-runs are instant.
dataset = load_dataset("gfissore/arxiv-abstracts-2021", split="train")

papers = []
# IDs are roughly chronological; iterate from the end to land on 2021/2020/2019 papers first.
for i in range(len(dataset) - 1, -1, -1):
    if len(papers) >= 20000:
        break
    row = dataset[i]
    if not row["abstract"] or not row["title"]:
        continue
    # categories arrive as space-joined strings (e.g. ["cs.LG cs.CV"]); split each entry.
    cats = [tok for entry in row["categories"] for tok in entry.split()]
    if not any(c in ML_CATEGORIES for c in cats):
        continue
    # Year lives in the YYMM prefix of new-format arXiv IDs ("2104.01234" -> 2021).
    arxiv_id = row["id"]
    if "/" in arxiv_id or "." not in arxiv_id:
        continue  # skip pre-2007 IDs like "math/0506001"
    if 2000 + int(arxiv_id[:2]) < 2018:
        continue
    papers.append({
        "arxiv_id": arxiv_id,
        "title": row["title"].strip(),
        "abstract": row["abstract"].strip(),
        "tags": cats,
    })
print(f"Loaded {len(papers)} papers")

## Schema

One Qdrant collection. Each point is a chunk. Each chunk holds four named vectors that we'll fuse at query time:

- `dense_chunk`: the chunk's own embedding (body content).
- `dense_title`: the paper title embedding (topical naming).
- `dense_abstract`: the paper abstract embedding (paper-level view).
- `sparse_title`: BM25 over the title (lexical matches on rare entity names, jargon, specific model or paper names).

Categories live in the `tags` payload with a keyword index, so queries can pre-filter by category.

`dense_title`, `dense_abstract`, and `sparse_title` are duplicated across every chunk of the same paper. That trades a bit of storage for one-shot query fusion (one collection, one Query API call, every representation reachable from any point). For the typical case (a few dozen chunks per paper, embeddings under a kilobyte each) it's the simpler choice.


In [ ]:
from qdrant_client import QdrantClient, models

# Replace url and api_key with your own from https://cloud.qdrant.io
client = QdrantClient(
    url="https://xyz-example.qdrant.io:6333",
    api_key="<your-api-key>",
    cloud_inference=True,
)

# 384 is the output dimension of sentence-transformers/all-minilm-l6-v2, used below for every dense vector.
client.create_collection(
    collection_name="arxiv_multi_repr",
    vectors_config={
        "dense_chunk":    models.VectorParams(size=384, distance=models.Distance.COSINE),
        "dense_title":    models.VectorParams(size=384, distance=models.Distance.COSINE),
        "dense_abstract": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse_title": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)

# Index 'document_id' so the Query API can group by it; index 'tags' so we can filter on category.
client.create_payload_index(
    collection_name="arxiv_multi_repr",
    field_name="document_id",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
client.create_payload_index(
    collection_name="arxiv_multi_repr",
    field_name="tags",
    field_schema=models.PayloadSchemaType.KEYWORD,
)


## Ingestion

Embeddings are generated server-side via Qdrant Cloud Inference:

- `sentence-transformers/all-minilm-l6-v2` (384-dim) for the three dense vectors.
- `qdrant/bm25` (core BM25 since Qdrant 1.15) for the sparse vector, with `avg_len=10.0` calibrated for the title-only field (default is 256, calibrated for document-length text).

Chunking uses a fixed two-sentence window for simplicity; the right chunking strategy depends on your document structure. One point per chunk, with the title and abstract Documents reused across every chunk of the same paper.


In [ ]:
DENSE_MODEL = "sentence-transformers/all-minilm-l6-v2"
BM25_MODEL = "qdrant/bm25"

def chunk_sentences(text, target_len=2):
    """Split text into ~2-sentence chunks; fall back to the full text if it doesn't split cleanly."""
    sentences = [s.strip() for s in text.split(". ") if s.strip()]
    return [". ".join(sentences[i:i + target_len])
            for i in range(0, len(sentences), target_len)] or [text]


points = []
for paper in papers:
    chunks = chunk_sentences(paper["abstract"])

    # Title, abstract, and sparse docs are reused across every chunk of this paper; only the chunk text varies.
    # Cloud Inference embeds each Document on the server, so you don't need a client-side embedding library.
    title_doc    = models.Document(text=paper["title"],    model=DENSE_MODEL)
    abstract_doc = models.Document(text=paper["abstract"], model=DENSE_MODEL)
    # avg_len is the average word count of the indexed text.
    # Default is 256 (document-length); setting it to the actual field length (~10 here) improves BM25 scoring accuracy.
    sparse_doc   = models.Document(
        text=paper["title"],
        model=BM25_MODEL,
        options={"avg_len": 10.0},
    )

    for i, chunk in enumerate(chunks):
        points.append(models.PointStruct(
            id=len(points),
            vector={
                "dense_chunk":    models.Document(text=chunk, model=DENSE_MODEL),
                "dense_title":    title_doc,
                "dense_abstract": abstract_doc,
                "sparse_title":   sparse_doc,
            },
            payload={
                "document_id": paper["arxiv_id"],
                "title":       paper["title"],
                "tags":        paper["tags"],
                "chunk_index": i,
                "chunk_text":  chunk,
            },
        ))

client.upload_points(collection_name="arxiv_multi_repr", points=points, batch_size=256, parallel=2)
print(f"Uploaded {len(points)} chunks across {len(papers)} papers")

## Query Helpers

Two pieces used by every step below:

- `SAMPLE_QUERY` is the single query we run through every step so we can watch the same query produce different results as capabilities are added.
- `show_results(retrieve_fn)` runs the retrieve function and prints the top 5 results: title, category tags, and an excerpt from the matching chunk. Accepts both chunk-level results (Steps 1-4) and grouped results (Steps 5-6, where each result is a paper with several chunks).


In [ ]:
import textwrap

SAMPLE_QUERY = "diffusion models for image synthesis"

def show_results(retrieve_fn, query=SAMPLE_QUERY, k=5):
    """Print top-k results as: title, category tags, and a matching-chunk excerpt."""
    print(f"Query: {query!r}\n")
    for i, item in enumerate(retrieve_fn(query, limit=k), 1):
        # item is a Point (Steps 1-4) or a Group (Steps 5-6).
        # For groups, hits[0] is the top chunk for that paper.
        point = item.hits[0] if hasattr(item, "hits") else item
        payload = point.payload
        title = payload["title"]
        tags = payload.get("tags", [])
        # Collapse whitespace (including embedded newlines) so the excerpt prints cleanly.
        chunk = " ".join(payload["chunk_text"].split())
        excerpt = chunk[:250].rstrip() + ("..." if len(chunk) > 250 else "")
        print(textwrap.fill(f"{i}. {title}", width=140, initial_indent="  ", subsequent_indent="     "))
        if tags:
            print(f"     [{', '.join(str(t) for t in tags[:3])}]")
        print(textwrap.fill(excerpt, width=140, initial_indent="     ", subsequent_indent="     "))
        print()


## Step 1: Dense Over Chunks (Baseline)

The naive baseline: encode the query with the dense model, search against `dense_chunk` only, return the chunk-level results' parent papers. No fusion, no title or sparse signal.

This is what most "vector search" tutorials stop at. It's a reasonable default for short, homogeneous corpora where the chunk text already carries the full signal. It systematically underperforms when the signal lives outside the chunk: in the title (topical naming), or in keyword overlap that the embedding model has averaged out into a generic neighborhood.

Each subsequent step closes one of those gaps.


In [ ]:
def retrieve_baseline(query, limit=10):
    return client.query_points(
        collection_name="arxiv_multi_repr",
        query=models.Document(text=query, model=DENSE_MODEL),
        using="dense_chunk",
        limit=limit,
    ).points

show_results(retrieve_baseline)


## Step 2: Add Sparse Title With RRF

Add a second prefetch: BM25 over the title. Then fuse the two ranked lists with **Reciprocal Rank Fusion (RRF)**.

Why RRF instead of weighted averages of raw scores? RRF works on rank, not score. Dense scores live in [0, 1], sparse BM25 scores don't, and RRF doesn't have to reconcile the two. Linear weights are fragile: a weight that helps one query class hurts another, and the right weight depends on query length, model, and corpus.

What does sparse add? Queries with rare entity names, jargon, or specific model/paper names often produce dense embeddings near generic neighborhoods. The sparse path catches those exact-token matches on the title. RRF promotes documents both paths agree on.


In [ ]:
def retrieve_hybrid(query, limit=10):
    dense_query  = models.Document(text=query, model=DENSE_MODEL)
    sparse_query = models.Document(text=query, model=BM25_MODEL)
    return client.query_points(
        collection_name="arxiv_multi_repr",
        prefetch=[
            models.Prefetch(query=dense_query,  using="dense_chunk", limit=50),
            models.Prefetch(query=sparse_query, using="sparse_title", limit=50),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
    ).points

show_results(retrieve_hybrid)


## Step 3: Add Title Prefetch

Add a third prefetch: the same dense query vector, but searched against `dense_title` instead of `dense_chunk`. We're now fusing across three representations: chunk content, title (lexical), and title (semantic).

The title prefetch saves queries where the topic is named explicitly but not echoed in any single chunk. For example: "diffusion models for high-resolution image synthesis" surfaces a paper titled "High-Resolution Image Synthesis with Latent Diffusion Models" via the title path even when its chunks phrase the contribution differently. The chunk prefetch alone misses it; the title path catches it; RRF promotes it because both paths agree.


In [ ]:
def retrieve_three_repr(query, limit=10):
    dense_query  = models.Document(text=query, model=DENSE_MODEL)
    sparse_query = models.Document(text=query, model=BM25_MODEL)
    return client.query_points(
        collection_name="arxiv_multi_repr",
        prefetch=[
            models.Prefetch(query=dense_query,  using="dense_chunk", limit=50),
            models.Prefetch(query=dense_query,  using="dense_title", limit=50),
            models.Prefetch(query=sparse_query, using="sparse_title", limit=50),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
    ).points

show_results(retrieve_three_repr)


## Step 4: Add Abstract Prefetch

Add a fourth prefetch on `dense_abstract`. The abstract gives a paper-level view that sits between the title (very short) and individual chunks (very local). It catches queries that match the paper's overall framing rather than a single passage or the title's topical naming.

In a production setup where chunks are full paper bodies, the abstract is a meaningfully different representation. In this notebook's arXiv dataset (where chunks are 2-sentence slices of the abstract itself), the lift over Step 3 will be smaller because the abstract and the chunks share text. The prefetch is still worth wiring up; the pipeline shape is what generalizes to longer corpora.


In [ ]:
def retrieve_four_repr(query, limit=10):
    dense_query  = models.Document(text=query, model=DENSE_MODEL)
    sparse_query = models.Document(text=query, model=BM25_MODEL)
    return client.query_points(
        collection_name="arxiv_multi_repr",
        prefetch=[
            models.Prefetch(query=dense_query,  using="dense_chunk",    limit=50),
            models.Prefetch(query=dense_query,  using="dense_title",    limit=50),
            models.Prefetch(query=dense_query,  using="dense_abstract", limit=50),
            models.Prefetch(query=sparse_query, using="sparse_title",   limit=50),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
    ).points

show_results(retrieve_four_repr)


## Step 5: Group by Document

So far results are chunks, and the same paper can appear multiple times in the top 10. Most consumers want one entry per document with the top chunks attached: a results UI, a citation list, an LLM that needs document-level attribution.

`query_points_groups` collapses chunks back to documents using `group_by="document_id"`. Each group's `hits` field carries the top-`group_size` chunks for that paper.

This step also wires in an optional `tags` parameter that filters candidates to specific arXiv categories before retrieval runs. Qdrant pre-filters on the payload index we added in the schema, so filtering happens before the fusion math, not after.

A few things worth knowing:

- Grouping is a *presentation* choice, not a relevance technique. The candidates and their fused scores don't change; only the result shape does.
- You may need to adjust the per-prefetch `limit` based on the number of chunks per document; grouping only sees what the prefetch returns.


In [ ]:
def retrieve_grouped(query, limit=10, group_size=3, tags=None):
    dense_query  = models.Document(text=query, model=DENSE_MODEL)
    sparse_query = models.Document(text=query, model=BM25_MODEL)
    # Optional category filter. When tags is provided, Qdrant pre-filters candidates
    # to points whose 'tags' payload includes any of the given values.
    query_filter = (
        models.Filter(must=[models.FieldCondition(key="tags", match=models.MatchAny(any=tags))])
        if tags else None
    )
    # query_points_groups runs the prefetches, fuses with RRF, applies the filter, and groups results by document_id.
    return client.query_points_groups(
        collection_name="arxiv_multi_repr",
        prefetch=[
            models.Prefetch(query=dense_query,  using="dense_chunk",    limit=100),
            models.Prefetch(query=dense_query,  using="dense_title",    limit=100),
            models.Prefetch(query=dense_query,  using="dense_abstract", limit=100),
            models.Prefetch(query=sparse_query, using="sparse_title",   limit=100),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        query_filter=query_filter,
        group_by="document_id",
        group_size=group_size,
        limit=limit,
    ).groups

show_results(retrieve_grouped)


## Step 6: Score Boosting With a Formula

When you have ranking preferences that aren't captured by similarity alone (recency, source authority, geographic proximity, structured boosts), swap RRF for a `FormulaQuery`. Formulas operate on the prefetch scores and payload fields:

- `$score[i]` references the score from prefetch `i`. Prefetch order is load-bearing.
- The `defaults` map provides fallback values for candidates that didn't appear in every prefetch, so the formula still evaluates.

The formula below sums the chunk score with weighted contributions from the title, abstract, and sparse prefetches. This is a linear combination of raw scores, which breaks down when prefetches use different scoring scales. RRF avoids this by discarding scores; DBSF normalizes per prefetch; a custom formula has to align distributions itself, typically with [decay functions](https://qdrant.tech/documentation/search/search-relevance/#decay-functions). The full FormulaQuery syntax lives in the [Score Boosting](https://qdrant.tech/documentation/search/search-relevance/#score-boosting) reference.

For time-based decay on a `published_at` payload field, swap a term for an `exp_decay` expression.

For RRF vs. DBSF guidance, see the [hybrid-search FAQ](https://qdrant.tech/documentation/faq/qdrant-fundamentals/#when-should-i-use-reciprocal-rank-fusion-rrf-vs-distribution-based-score-fusion-dbsf-for-hybrid-search).


In [ ]:
def retrieve_boosted(query, limit=10, group_size=3):
    dense_query  = models.Document(text=query, model=DENSE_MODEL)
    sparse_query = models.Document(text=query, model=BM25_MODEL)
    return client.query_points_groups(
        collection_name="arxiv_multi_repr",
        prefetch=[
            # $score[0] = chunk, $score[1] = title, $score[2] = abstract, $score[3] = sparse
            models.Prefetch(query=dense_query,  using="dense_chunk",    limit=100),
            models.Prefetch(query=dense_query,  using="dense_title",    limit=100),
            models.Prefetch(query=dense_query,  using="dense_abstract", limit=100),
            models.Prefetch(query=sparse_query, using="sparse_title",   limit=100),
        ],
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=[
                models.MultExpression(mult=[1.0, "$score[0]"]),
                models.MultExpression(mult=[0.5, "$score[1]"]),
                models.MultExpression(mult=[0.4, "$score[2]"]),
                models.MultExpression(mult=[0.3, "$score[3]"]),
            ]),
            defaults={"$score[1]": 0.0, "$score[2]": 0.0, "$score[3]": 0.0},
        ),
        group_by="document_id",
        group_size=group_size,
        limit=limit,
    ).groups

show_results(retrieve_boosted)


## Wrap-up

That's the recommended multi-representation pipeline end to end. The same schema works for any corpus with title-like, abstract-like, and body-like representations.

If you ran this notebook with the same `SAMPLE_QUERY` ("diffusion models for image synthesis") and the same 20,000-paper arXiv slice, here's roughly what each step's top 5 should produce:

- **Step 1 (`dense_chunk` only):** chunk-level results with the same paper appearing in multiple slots. SegDiff, LDM, GLIDE in the top 5.
- **Step 2 (+ `sparse_title`):** title-exact matches surface. Vector Quantized Diffusion Model jumps in.
- **Step 3 (+ `dense_title`):** LDM dominates with three of its own chunks. Semantic title match takes over.
- **Step 4 (+ `dense_abstract`):** modest shift. GLIDE returns thanks to abstract-level signal. Adding a prefetch isn't always dramatic.
- **Step 5 (grouping):** one entry per paper. The collapsed LDM chunks free up slots for Palette, Global Context, and Implicit Image Segmentation.
- **Step 6 (formula):** custom weighting reorders results. Vector Quantized Diffusion climbs back; ImageBART and Manifold-aware Synthesis enter as the formula amplifies raw scores differently from RRF's rank-based fusion.

Swap the dataset, retune which representations earn their prefetch slots for your data, and wire in formula-based ranking preferences as needed.

For the design rationale and references, see the [tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/multi-representation-search/).
